# **Task 2 — DeiT**: does seeing more pixels help?

This week's seminar was about [DeiT](https://proceedings.mlr.press/v139/touvron21a.html): a ViT trained on ImageNet alone, with distillation, and fine-tuned at a higher resolution at the end. The paper reports that going from 224×224 to 384×384 images makes the model more accurate (Table 10). Today you check whether that holds for **our** model and **our** data.

This notebook loads the data and the model and gives you a **fixed evaluation**, so that every group measures its result the same way. Your own work starts after the baseline.

<details>
<summary>Reach out here if you have any problems</summary>
Taras Kutsyk — taras.kutsyk@doctoral.uj.edu.pl
</details>

# The task

**Question.** Does seeing more pixels help? Make the model run at 384×384, fine-tune it within the budget, and **answer the question for this model and this data, with evidence**. Then explain *why* you got the result you got, and what they teach you.

*Hint:* be wary of what you compare with, and of how your claim relates to the paper's.

**Fixed:**
- the starting model and the dataset;
- the `evaluate()` function below;
- the fine-tuning budget: **at most 100,000 training images per fine-tuned model** (about 11 passes over the training set);
- write the resizing of the position embeddings yourselves: no ready-made resizing (e.g. timm's `img_size=` argument or `resample_abs_pos_embed`).

Everything else is your choice.

**Bring to the lab:** your code + a short deck with your answer, the evidence for it, and your explanation. If your group defends this week, **every member should be able to explain any piece of the code**.

# Setup

**Step 0.** Runtime → Change runtime type → **T4 GPU**. Everything we need (PyTorch, torchvision, timm) is already installed on Colab.

**Step 1. Data.** [Imagewoof](https://github.com/fastai/imagenette): 10 dog breeds from ImageNet, 9,025 training and 3,929 validation photos, **at full size** (about 1.3 GB, under a minute to download).

In [ ]:
import os, tarfile, time, urllib.request, copy
import torch, timm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch to a GPU runtime: Runtime -> Change runtime type -> T4 GPU"

URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2.tgz"
# we use the full-size image sets on purpose, not the smaller "-320" versions
# can you tell why? (in the deck)
if not os.path.exists("imagewoof2/val"):
    urllib.request.urlretrieve(URL, "imagewoof2.tgz")
    with tarfile.open("imagewoof2.tgz") as t:
        t.extractall(filter="data")
for split in ["train", "val"]:
    print(split, sum(len(f) for _, _, f in os.walk(f"imagewoof2/{split}")), "photos")

train 9025 photos
val 3929 photos


**Step 2. Model.** A distilled DeiT-S from the paper (12 blocks, 384 dims, 22M parameters, the DeiT-S⚗ of Table 5), trained on ImageNet at 224×224, together with the exact preprocessing it was trained with.

The model predicts one of 1000 ImageNet classes, so we map each Imagewoof folder (e.g. `n02086240` = Shih-Tzu) to its **ImageNet index**: the position of that breed among the model's 1000 outputs. You will need `to_imagenet` for your fine-tuning too.

<details>
<summary>Running the cell below can ask for your HF_TOKEN secret</summary>
Just press "Grant Access", it's automatically generated by Colab
</details>

In [ ]:
model = timm.create_model("deit_small_distilled_patch16_224", pretrained=True).to(device).eval()
cfg = timm.data.resolve_data_config({}, model=model)  # the preprocessing the model was trained with

# ImageFolder labels the 10 folders 0..9, but the model outputs 1000 ImageNet classes:
# to_imagenet[label] translates a folder label into the ImageNet class index
wnids = list(timm.data.ImageNetInfo().label_names())
breeds = datasets.ImageFolder("imagewoof2/val").classes
to_imagenet = torch.tensor([wnids.index(w) for w in breeds], device=device)
imagenet_to_woof = {int(to_imagenet[i]): i for i in range(len(to_imagenet))}
print(cfg)
print("folder -> ImageNet index:", dict(zip(breeds, to_imagenet.tolist())))

{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.9, 'crop_mode': 'center'}
folder -> ImageNet index: {'n02086240': 155, 'n02087394': 159, 'n02088364': 162, 'n02089973': 167, 'n02093754': 182, 'n02096294': 193, 'n02099601': 207, 'n02105641': 229, 'n02111889': 258, 'n02115641': 273}


**Step 3. The fixed evaluation** (DON'T modify it). `evaluate(model, resolution)` resizes the validation photos for the given resolution exactly as the model's own preprocessing does, and returns the **top-1 accuracy over the 10 breeds**: the model's prediction is the highest-scoring of its 10 dog outputs.

In [ ]:
@torch.no_grad()
def evaluate(model, resolution):
    """The fixed evaluation for task 2. Do not change it; report exactly what it prints.

    model      - any model that takes (batch, 3, resolution, resolution) images and returns logits
    resolution - the input size: 224 for the original model, e.g. 384 for yours
    returns: accuracy - top-1 over the 10 breeds, on the 3,929 Imagewoof validation photos
    """
    model.eval()
    tf = timm.data.create_transform(**dict(cfg, input_size=(3, resolution, resolution)))
    loader = torch.utils.data.DataLoader(datasets.ImageFolder("imagewoof2/val", transform=tf),
                                         batch_size=64, num_workers=2)
    correct = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            if isinstance(logits, (tuple, list)):
                logits = (logits[0] + logits[1]) * 0.5
            # Support both adapted 10-class models and standard 1000-class models
            pred = logits.argmax(-1) if logits.shape[-1] == 10 else logits[:, to_imagenet].argmax(-1)
        correct += (pred == y).sum().item()
        n += len(y)
    return correct / n

**Step 4. Baseline.** Takes about half a minute. You should get **top-1 ≈ 97.20%** at 224. If not, something in the setup went wrong.

In [ ]:
print(f"baseline DeiT-S distilled @224: top-1 {evaluate(model, 224):.2%}")

baseline DeiT-S distilled @224: top-1 97.20%


# Before you start

Think about **what exactly breaks in the model when you pass it higher-resolution images?** What's the simplest, most effective way to fix that?

Training photos are in `imagewoof2/train` (same folder structure as `val`).

# Your solution

Make the model run at 384, fine-tune within the budget, and collect the evidence for your answer. Good luck!

--- 
### Linear Probe (B1, B2) & Layer Freezing (D) @ 384
Exploring whether parameter-efficient fine-tuning allows DeiT-S to exploit 384×384 pixels without full model retraining.

In [ ]:
# ── Positional Embedding Interpolation & Unified Training Engine ────────────
RES = 384
MAX_EPOCHS = 11  # 11 * 9,025 = 99,275 images (within <= 100,000 budget limit)
BATCH_SIZE = 32

def interpolate_pos_embed(model_to_modify, new_res=RES):
    """Resizes spatial PE from 14x14 (224) to 24x24 (384), keeping [CLS] and [DIST] tokens."""
    new_grid = new_res // 16  # 24
    pe = model_to_modify.pos_embed.data
    cls_dist_pe = pe[:, :2, :]          # [CLS] and [DIST]
    patch_pe = pe[:, 2:, :]             # [1, 196, 384]
    D = patch_pe.shape[-1]
    old_grid = int(patch_pe.shape[1] ** 0.5)  # 14

    patch_pe = patch_pe.reshape(1, old_grid, old_grid, D).permute(0, 3, 1, 2)
    patch_pe = F.interpolate(patch_pe.float(), size=(new_grid, new_grid), mode="bicubic", align_corners=False)
    patch_pe = patch_pe.permute(0, 2, 3, 1).reshape(1, new_grid * new_grid, D)

    model_to_modify.pos_embed = nn.Parameter(torch.cat([cls_dist_pe, patch_pe], dim=1))
    model_to_modify.patch_embed.img_size = (new_res, new_res)
    model_to_modify.patch_embed.grid_size = (new_grid, new_grid)
    model_to_modify.patch_embed.num_patches = new_grid * new_grid
    return model_to_modify

# Data loader for 384x384 training
tf_train_384 = timm.data.create_transform(**dict(cfg, input_size=(3, RES, RES), is_training=True))
train_loader_384 = DataLoader(
    datasets.ImageFolder("imagewoof2/train", transform=tf_train_384),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)

def train_model(model_under_test, optimizer, label, teacher=None, epochs=MAX_EPOCHS):
    """Unified, idiomatic training loop with AMP FP16, grad clipping, and cosine annealing."""
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")
    ce_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)
    ce_hard = nn.CrossEntropyLoss()

    trainable = sum(p.numel() for p in model_under_test.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model_under_test.parameters())
    print(f"[{label}] Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

    best_acc = 0.0
    best_weights = None
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model_under_test.train()
        total_loss = 0.0

        for x, y in train_loader_384:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                out = model_under_test(x)
                cls_logits, dist_logits = out if isinstance(out, tuple) else (out, out)

                if teacher is not None:
                    with torch.no_grad():
                        x_224 = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
                        t_logits = teacher(x_224)
                        t_preds = t_logits.argmax(dim=-1).cpu().tolist()
                        t_woof = torch.tensor([imagenet_to_woof.get(idx, int(y[i])) for i, idx in enumerate(t_preds)],
                                              dtype=torch.long, device=device)
                    loss = 0.5 * ce_smooth(cls_logits, y) + 0.5 * ce_hard(dist_logits, t_woof)
                else:
                    loss = 0.5 * ce_smooth(cls_logits, y) + 0.5 * ce_smooth(dist_logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_under_test.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        scheduler.step()
        val_acc = evaluate(model_under_test, RES) * 100.0
        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = copy.deepcopy(model_under_test.state_dict())

        avg_loss = total_loss / len(train_loader_384)
        elapsed = (time.time() - t0) / 60
        print(f"[{label}] ep {ep:2d}/{epochs} | loss={avg_loss:.4f} | val={val_acc:.2f}% | best={best_acc:.2f}% ({elapsed:.1f}m)")

    model_under_test.load_state_dict(best_weights)
    print(f"→ [{label}] Best validation accuracy: {best_acc:.2f}%\n")
    return best_acc

In [ ]:
# ════════════════════════════════════════════════════════════
#  SOLUTION B1  —  Linear Probe @ 384
#  Trainable: head + head_dist  (≈ 2×384×10 = 7 680 params)
#  Backbone + pos_embed: fully frozen
# ════════════════════════════════════════════════════════════

print("=" * 56)
print("SOLUTION B1 — Linear Probe (head + head_dist only) @ 384")
print("=" * 56)

model_b1 = timm.create_model("deit_small_distilled_patch16_224", pretrained=True, num_classes=10).to(device)
interpolate_pos_embed(model_b1)

# Freeze all backbone layers and pos_embed; unfreeze only classification heads
model_b1.requires_grad_(False)
model_b1.head.requires_grad_(True)
model_b1.head_dist.requires_grad_(True)

opt_b1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_b1.parameters()), lr=1e-3, weight_decay=0.05)
acc_b1 = train_model(model_b1, opt_b1, label="B1")

SOLUTION B1 — Linear Probe (head + head_dist only) @ 384
[B1] Trainable params: 7,700 / 21,820,052 (0.04%)
[B1] ep  1/11 | loss=0.7964 | val=96.54% | best=96.54% (3.7m)
[B1] ep  2/11 | loss=0.7237 | val=96.79% | best=96.79% (7.6m)
[B1] ep  3/11 | loss=0.7202 | val=96.72% | best=96.79% (11.4m)
[B1] ep  4/11 | loss=0.7084 | val=96.69% | best=96.79% (15.5m)
[B1] ep  5/11 | loss=0.7060 | val=96.77% | best=96.79% (19.4m)
[B1] ep  6/11 | loss=0.6984 | val=96.72% | best=96.79% (23.3m)
[B1] ep  7/11 | loss=0.6938 | val=96.97% | best=96.97% (27.2m)
[B1] ep  8/11 | loss=0.6949 | val=97.12% | best=97.12% (31.2m)
[B1] ep  9/11 | loss=0.6874 | val=97.00% | best=97.12% (35.1m)
[B1] ep 10/11 | loss=0.6967 | val=97.00% | best=97.12% (39.1m)
[B1] ep 11/11 | loss=0.6869 | val=97.05% | best=97.12% (43.1m)
→ [B1] Best validation accuracy: 97.12%



In [ ]:
# ── Solution B2: Linear Probe + pos_embed @ 384 ─────────────────────────────
print("=" * 56)
print("SOLUTION B2 — Linear Probe + pos_embed @ 384")
print("=" * 56)

model_b2 = timm.create_model("deit_small_distilled_patch16_224", pretrained=True, num_classes=10).to(device)
interpolate_pos_embed(model_b2)

# Unfreeze heads and pos_embed, leaving all 12 transformer blocks frozen
model_b2.requires_grad_(False)
model_b2.head.requires_grad_(True)
model_b2.head_dist.requires_grad_(True)
model_b2.pos_embed.requires_grad_(True)

# Differential learning rates: 1e-4 for pos_embed (coordinate calibration), 1e-3 for heads
opt_b2 = torch.optim.AdamW([
    {"params": model_b2.head.parameters(), "lr": 1e-3},
    {"params": model_b2.head_dist.parameters(), "lr": 1e-3},
    {"params": [model_b2.pos_embed], "lr": 1e-4},
], weight_decay=0.05)

acc_b2 = train_model(model_b2, opt_b2, label="B2")

SOLUTION B2 — Linear Probe + pos_embed @ 384
[B2] Trainable params: 229,652 / 21,820,052 (1.05%)
[B2] ep  1/11 | loss=0.7865 | val=96.72% | best=96.72% (4.0m)
[B2] ep  2/11 | loss=0.7111 | val=96.87% | best=96.87% (8.0m)
[B2] ep  3/11 | loss=0.6966 | val=97.07% | best=97.07% (12.1m)
[B2] ep  4/11 | loss=0.6870 | val=96.89% | best=97.07% (16.0m)
[B2] ep  5/11 | loss=0.6866 | val=96.77% | best=97.07% (20.1m)
[B2] ep  6/11 | loss=0.6812 | val=96.82% | best=97.07% (24.1m)
[B2] ep  7/11 | loss=0.6798 | val=96.74% | best=97.07% (28.1m)
[B2] ep  8/11 | loss=0.6709 | val=97.02% | best=97.07% (32.2m)
[B2] ep  9/11 | loss=0.6692 | val=97.02% | best=97.07% (36.4m)
[B2] ep 10/11 | loss=0.6726 | val=97.00% | best=97.07% (40.5m)
[B2] ep 11/11 | loss=0.6694 | val=97.02% | best=97.07% (44.5m)
→ [B2] Best validation accuracy: 97.07%



In [ ]:
# ════════════════════════════════════════════════════════════
#  SOLUTION D  —  Layer Freezing @ 384
#
#  Architecture:
#    patch_embed + blocks 0–7  → FROZEN   (8 / 12 blocks)
#    pos_embed + blocks 8–11   → trainable
#    norm + head + head_dist   → trainable
#
#  Loss: Hard Knowledge Distillation
#    Teacher = DeiT-S @224 (pretrained, 1 000 classes, frozen)
#    Student = DeiT-S @384 (blocks 8–11 + PE + heads trainable)
#
#  WHY freeze early blocks?
#    Blocks 0–7 learn low-level features (edges, textures, local
#    patches) that are largely resolution-agnostic and already
#    well-trained on ImageNet.  Keeping them frozen:
#      (a) cuts trainable params → far less overfitting risk
#          on our 9 025-image dataset;
#      (b) preserves the strong ImageNet representations.
#    Blocks 8–11 learn high-level, global semantics that benefit
#    from 384×384's extra context — they need to adapt.
# ════════════════════════════════════════════════════════════

print("=" * 56)
print("SOLUTION D — Layer Freezing (blocks 8–11 + PE + heads)")
print("=" * 56)

# Teacher: Pretrained DeiT-S @224, 1000 classes, frozen
teacher = timm.create_model("deit_small_distilled_patch16_224", pretrained=True, num_classes=1000).to(device).eval()
teacher.requires_grad_(False)

# Student model
model_d = timm.create_model("deit_small_distilled_patch16_224", pretrained=True, num_classes=10).to(device)
interpolate_pos_embed(model_d)

# Freeze patch_embed, CLS, DIST, and early blocks 0–7; unfreeze blocks 8–11 + norm + heads + pos_embed
model_d.requires_grad_(False)
for blk in model_d.blocks[8:]:
    blk.requires_grad_(True)
model_d.pos_embed.requires_grad_(True)
model_d.norm.requires_grad_(True)
model_d.head.requires_grad_(True)
model_d.head_dist.requires_grad_(True)

opt_d = torch.optim.AdamW([
    {"params": [model_d.pos_embed], "lr": 5e-5},
    {"params": model_d.norm.parameters(), "lr": 5e-5},
    {"params": [p for blk in model_d.blocks[8:] for p in blk.parameters()], "lr": 5e-5},
    {"params": model_d.head.parameters(), "lr": 5e-4},
    {"params": model_d.head_dist.parameters(), "lr": 5e-4},
], weight_decay=0.05)

acc_d = train_model(model_d, opt_d, label="D", teacher=teacher)

SOLUTION D — Layer Freezing (blocks 8–11 + PE + heads)
[D] Trainable params: 7,328,276 / 21,820,052 (33.59%)
[D] ep  1/11 | loss=0.5832 | val=96.26% | best=96.26% (4.2m)
[D] ep  2/11 | loss=0.4648 | val=96.03% | best=96.26% (8.3m)
[D] ep  3/11 | loss=0.4304 | val=96.74% | best=96.74% (12.5m)
[D] ep  4/11 | loss=0.4286 | val=96.79% | best=96.79% (16.6m)
[D] ep  5/11 | loss=0.4222 | val=96.61% | best=96.79% (20.9m)
[D] ep  6/11 | loss=0.4132 | val=97.02% | best=97.02% (25.0m)
[D] ep  7/11 | loss=0.3997 | val=96.95% | best=97.02% (29.0m)
[D] ep  8/11 | loss=0.3966 | val=96.95% | best=97.02% (32.9m)
[D] ep  9/11 | loss=0.3910 | val=97.15% | best=97.15% (37.0m)
[D] ep 10/11 | loss=0.3938 | val=97.05% | best=97.15% (41.1m)
[D] ep 11/11 | loss=0.3827 | val=97.02% | best=97.15% (45.2m)
→ [D] Best validation accuracy: 97.15%



In [ ]:
# ════════════════════════════════════════════════════════════
#  RESULTS SUMMARY
# ════════════════════════════════════════════════════════════
print("=" * 56)
print("RESULTS SUMMARY")
print("=" * 56)
BASELINE = 97.20   # starter notebook: DeiT-S @224, no FT

rows = [
    ("Baseline DeiT-S @224 (no FT)",          BASELINE),
    ("B1  Linear probe @384 (head+head_dist)", acc_b1),
    ("B2  Linear probe @384 (+pos_embed)",     acc_b2),
    ("D   Layer-frozen FT @384 (blks 8–11)",   acc_d),
]
for name, acc in rows:
    delta = acc - BASELINE
    sign  = "+" if delta >= 0 else ""
    print(f"  {name:<42s}  {acc:.2f}%  ({sign}{delta:.2f}%)")

print(f"\n  Budget used: {MAX_EPOCHS * 9025:,} / 100,000 images")
print("  Does seeing more pixels help?")
best = max(acc_b1, acc_b2, acc_d)
print(f"  Best @384: {best:.2f}%  vs baseline @224: {BASELINE:.2f}%  "
      f"→ {'YES ✓' if best > BASELINE else 'NO ✗'}")

RESULTS SUMMARY
  Baseline DeiT-S @224 (no FT)                97.20%  (+0.00%)
  B1  Linear probe @384 (head+head_dist)      97.12%  (-0.08%)
  B2  Linear probe @384 (+pos_embed)          97.07%  (-0.13%)
  D   Layer-frozen FT @384 (blks 8–11)        97.15%  (-0.05%)

  Budget used: 99,275 / 100,000 images
  Does seeing more pixels help?
  Best @384: 97.15%  vs baseline @224: 97.20%  → NO ✗
